In [1]:
import pandas as pd
import joblib
import numpy as np

In [2]:
credit_score = pd.read_csv('../../creditscore_anon_cw_july2023.csv')
credit_score.head()

,A0,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,1,4,31,Female,5,3,1,3,0,1,...,0,4,0,0,0,0,0,0,0,28553.30
1,2,4,31,Female,5,3,1,3,0,1,...,0,2,0,0,0,0,0,0,0,10410.00
2,3,3,26,Female,7,3,1,0,0,1,...,0,7,0,104,0,0,0,0,0,10979.06
3,4,0,63,Male,7,3,3,2,0,2,...,0,3,0,0,0,0,0,0,0,10242.00
4,5,0,26,Male,5,3,1,1,0,1,...,1,0,0,0,0,0,0,0,200,1489.00


# Data cleaning

The dataset seems to include a column A0 which is the ID or the row that wa can get rid of, it doesnt bring anything

In [3]:
credit_score = credit_score.drop(columns=['A0'], axis=1)

In [4]:
credit_score.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,4,31,Female,5,3,1,3,0,1,24,...,0,4,0,0,0,0,0,0,0,28553.30
1,4,31,Female,5,3,1,3,0,1,24,...,0,2,0,0,0,0,0,0,0,10410.00
2,3,26,Female,7,3,1,0,0,1,24,...,0,7,0,104,0,0,0,0,0,10979.06
3,0,63,Male,7,3,3,2,0,2,3,...,0,3,0,0,0,0,0,0,0,10242.00
4,0,26,Male,5,3,1,1,0,1,24,...,1,0,0,0,0,0,0,0,200,1489.00


# Pre-processing Pipeline

Indetifying the categorical and the numerical columns

In [5]:
categorical_columns = ['A3', 'A4', 'A5', 'A9', 'A10', 'A12', 'A16', 'A17', 'A18', 'A19', 'A20'] # Acccording to the dataset description

nominal_columns = ['A3', 'A5', 'A9', 'A10', 'A12', 'A16', 'A18', 'A19', 'A20'] # To one hot encode
ordinal_columns = [col for col in categorical_columns if col not in nominal_columns]
numerical_columns = [col for col in credit_score.columns if col not in categorical_columns and col != 'A1']

print("All categorical columns: %s" % categorical_columns)
print("Nomimal: %s" % nominal_columns)
print("Ordinal: %s" % ordinal_columns)

print("Numrical: %s" % numerical_columns)

All categorical columns: ['A3', 'A4', 'A5', 'A9', 'A10', 'A12', 'A16', 'A17', 'A18', 'A19', 'A20']
Nomimal: ['A3', 'A5', 'A9', 'A10', 'A12', 'A16', 'A18', 'A19', 'A20']
Ordinal: ['A4', 'A17']
Numrical: ['A2', 'A6', 'A7', 'A8', 'A11', 'A13', 'A14', 'A15', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45']


In [6]:
print("All Categorical")
display(credit_score[categorical_columns])

print("Nominal")
display(credit_score[nominal_columns])

print("Ordinal")
display(credit_score[ordinal_columns])

print("Numerical")
display(credit_score[numerical_columns])

All Categorical


,A3,A4,A5,A9,A10,A12,A16,A17,A18,A19,A20
0,Female,5,3,1,24,3,1,3,2,1,5
1,Female,5,3,1,24,3,1,3,2,1,5
2,Female,7,3,1,24,5,3,2,3,1,5
3,Male,7,3,2,3,3,1,3,2,1,1
4,Male,5,3,1,24,4,3,2,3,1,5
...,...,...,...,...,...,...,...,...,...,...,...
1862,Male,5,3,1,24,5,3,2,3,1,5
1863,Female,3,2,2,24,5,3,2,3,1,5
1864,Female,5,3,1,12,5,3,2,3,1,1
1865,Male,6,2,2,36,3,3,2,3,3,1


Nominal


,A3,A5,A9,A10,A12,A16,A18,A19,A20
0,Female,3,1,24,3,1,2,1,5
1,Female,3,1,24,3,1,2,1,5
2,Female,3,1,24,5,3,3,1,5
3,Male,3,2,3,3,1,2,1,1
4,Male,3,1,24,4,3,3,1,5
...,...,...,...,...,...,...,...,...,...
1862,Male,3,1,24,5,3,3,1,5
1863,Female,2,2,24,5,3,3,1,5
1864,Female,3,1,12,5,3,3,1,1
1865,Male,2,2,36,3,3,3,3,1


Ordinal


,A4,A17
0,5,3
1,5,3
2,7,2
3,7,3
4,5,2
...,...,...
1862,5,2
1863,3,2
1864,5,2
1865,6,2


Numerical


,A2,A6,A7,A8,A11,A13,A14,A15,A21,A22,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,31,1,3,0,2.0,0.0,60000.00,100000,0,0,...,0,4,0,0,0,0,0,0,0,28553.30
1,31,1,3,0,2.0,0.0,60000.00,50000,0,0,...,0,2,0,0,0,0,0,0,0,10410.00
2,26,1,0,0,5.0,1.6,30000.00,100000,0,0,...,0,7,0,104,0,0,0,0,0,10979.06
3,63,3,2,0,7.0,0.0,14866.67,20000,0,0,...,0,3,0,0,0,0,0,0,0,10242.00
4,26,1,1,0,1.0,2.0,17000.00,100000,0,0,...,1,0,0,0,0,0,0,0,200,1489.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1862,23,1,0,0,4.0,3.1,26000.00,50000,0,0,...,3,0,0,83,0,0,0,0,500,4846.00
1863,23,2,1,1,5.0,1.8,10000.00,57000,0,0,...,3,0,0,13,0,1,0,0,0,716.00
1864,35,1,1,0,12.0,10.9,24026.00,44000,0,0,...,13,12,0,106,5,0,3,4,1200,16384.09
1865,35,2,1,1,10.0,0.6,136819.00,200000,0,0,...,16,17,0,103,0,1,2,9,467,79824.00


Some numerical columns are skewed to we are going to fix it using log-transforming

In [7]:
skewed_cols = []
for col in numerical_columns:
    skew = credit_score[col].skew()
    if abs(skew) > 1:
        skewed_cols.append(col)
 
print("Applying log1p to skewed columns:", skewed_cols)
 
for col in skewed_cols:
    credit_score[col] = np.log1p(credit_score[col].clip(lower=0))

Applying log1p to skewed columns: ['A6', 'A7', 'A8', 'A11', 'A13', 'A14', 'A15', 'A23', 'A24', 'A25', 'A26', 'A27', 'A29', 'A30', 'A32', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45']


In [8]:
credit_score[numerical_columns]

,A2,A6,A7,A8,A11,A13,A14,A15,A21,A22,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,31,0.693147,1.386294,0.000000,1.098612,0.000000,11.002117,11.512935,0,0,...,0.000000,1.609438,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10.259563
1,31,0.693147,1.386294,0.000000,1.098612,0.000000,11.002117,10.819798,0,0,...,0.000000,1.098612,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.250618
2,26,0.693147,0.000000,0.000000,1.791759,0.955511,10.308986,11.512935,0,0,...,0.000000,2.079442,0.0,4.653960,0.000000,0.000000,0.000000,0.000000,0.000000,9.303836
3,63,1.386294,1.098612,0.000000,2.079442,0.000000,9.606944,9.903538,0,0,...,0.000000,1.386294,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.234350
4,26,0.693147,0.693147,0.000000,0.693147,1.098612,9.741027,11.512935,0,0,...,0.693147,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,5.303305,7.306531
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1862,23,0.693147,0.000000,0.000000,1.609438,1.410987,10.165890,10.819798,0,0,...,1.386294,0.000000,0.0,4.430817,0.000000,0.000000,0.000000,0.000000,6.216606,8.486115
1863,23,1.098612,0.693147,0.693147,1.791759,1.029619,9.210440,10.950824,0,0,...,1.386294,0.000000,0.0,2.639057,0.000000,0.693147,0.000000,0.000000,0.000000,6.575076
1864,35,0.693147,0.693147,0.000000,2.564949,2.476538,10.086933,10.691968,0,0,...,2.639057,2.564949,0.0,4.672829,1.791759,0.000000,1.386294,1.609438,7.090910,9.704127
1865,35,1.098612,0.693147,0.693147,2.397895,0.470004,11.826421,12.206078,0,0,...,2.833213,2.890372,0.0,4.644391,0.000000,0.693147,1.098612,2.302585,6.148468,11.287592


Making the pre-processing pipeline

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [10]:
credit_score_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
        ('nom', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), nominal_columns)
    ]
)

credit_score_preprocessor

,transformers,"[('num', ...), ('nom', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [11]:
joblib.dump(credit_score_preprocessor, './Data/credit_score_preprocessor.pkl')

['./Data/credit_score_preprocessor.pkl']

Saving the preprocessed data into rated and unrated

In [13]:
rated_credit_score = credit_score[credit_score['A1'] != 0]
unrated_credit_score = credit_score[credit_score['A1'] != 0]

In [14]:
rated_credit_score.to_csv("./Data/rated_credit_score.csv")
unrated_credit_score.to_csv("./Data/unrated_credit_score.csv")